# HI 5304 — Functions and OOP in Python for Health Informatics

This notebook demonstrates how **functions** and **object-oriented programming (OOP)** help you write clean, reusable code for health informatics tasks (rules, patient records, vitals, and simple registries).

**You’ll learn**
- Functions with parameters + return values
- Docstrings + simple validation
- Classes using `@dataclass`
- Methods + attributes
- A mini “follow-up list” workflow


## 1) Setup

In [ ]:
from dataclasses import dataclass, field
from datetime import date
from statistics import mean
from typing import List, Dict, Optional


## 2) Functions: reusable clinical rules

In health informatics, we often translate guideline logic into small functions:
- categorize BP
- flag controlled vs uncontrolled
- compute adherence


### 2.1 Blood pressure category

In [ ]:
def bp_category(systolic: int, diastolic: int) -> str:
    """Return a simplified BP category (teaching example)."""
    if systolic < 120 and diastolic < 80:
        return "Normal"
    elif systolic < 130 and diastolic < 80:
        return "Elevated"
    elif systolic < 140 or diastolic < 90:
        return "HTN Stage 1"
    else:
        return "HTN Stage 2"


for sys, dia in [(118, 76), (126, 78), (136, 86), (155, 95)]:
    print(f"{sys}/{dia} → {bp_category(sys, dia)}")


### 2.2 BP control flag

In [ ]:
def bp_controlled(systolic: int, diastolic: int, goal_sys: int = 130, goal_dia: int = 80) -> bool:
    """Return True if BP is controlled under a simple goal threshold."""
    return (systolic < goal_sys) and (diastolic < goal_dia)

print(bp_controlled(128, 78))   # True
print(bp_controlled(142, 85))   # False


### 2.3 Medication adherence rate

In [ ]:
def adherence_rate(days_taken: int, days_prescribed: int) -> float:
    """Compute adherence as a proportion (0–1)."""
    if days_prescribed <= 0:
        raise ValueError("days_prescribed must be > 0")
    if days_taken < 0:
        raise ValueError("days_taken must be >= 0")
    return days_taken / days_prescribed

print("Adherence:", round(adherence_rate(26, 30), 2))


## 3) OOP: modeling health concepts with classes

When projects grow, it helps to group data + logic together:
- `VitalsReading` stores one measurement and provides helper methods
- `Patient` stores demographics + lists of vitals, meds, conditions, and provides summary methods


## 4) Classes with `@dataclass`

### 4.1 `VitalsReading`: one measurement event

In [ ]:
@dataclass(frozen=True)
class VitalsReading:
    reading_date: date
    systolic: int
    diastolic: int
    heart_rate: int
    glucose: Optional[int] = None

    def bp_category(self) -> str:
        return bp_category(self.systolic, self.diastolic)

    def bp_controlled(self, goal_sys: int = 130, goal_dia: int = 80) -> bool:
        return bp_controlled(self.systolic, self.diastolic, goal_sys, goal_dia)


vr = VitalsReading(date(2026, 3, 5), systolic=128, diastolic=82, heart_rate=72, glucose=110)
print(vr)
print("Category:", vr.bp_category())
print("Controlled:", vr.bp_controlled())


### 4.2 `Patient`: a patient record with summary methods

In [ ]:
@dataclass
class Patient:
    patient_id: str
    name: str
    age: int
    sex: str
    conditions: List[str] = field(default_factory=list)
    medications: List[str] = field(default_factory=list)
    vitals: List[VitalsReading] = field(default_factory=list)

    def add_vital(self, reading: VitalsReading) -> None:
        self.vitals.append(reading)

    def latest_vital(self) -> Optional[VitalsReading]:
        if not self.vitals:
            return None
        return max(self.vitals, key=lambda r: r.reading_date)

    def average_systolic(self) -> Optional[float]:
        if not self.vitals:
            return None
        return mean([r.systolic for r in self.vitals])

    def bp_control_rate(self, goal_sys: int = 130, goal_dia: int = 80) -> Optional[float]:
        if not self.vitals:
            return None
        flags = [r.bp_controlled(goal_sys, goal_dia) for r in self.vitals]
        return sum(flags) / len(flags)

    def risk_flags(self) -> Dict[str, bool]:
        latest = self.latest_vital()
        return {
            "age_65_plus": self.age >= 65,
            "has_diabetes_condition": "diabetes" in [c.lower() for c in self.conditions],
            "latest_bp_uncontrolled": (latest is not None) and (not latest.bp_controlled()),
        }


p = Patient(
    patient_id="P10023",
    name="Jordan Lee",
    age=63,
    sex="F",
    conditions=["Hypertension", "Hyperlipidemia"],
    medications=["lisinopril", "atorvastatin"],
)

p.add_vital(VitalsReading(date(2026, 3, 1), 142, 90, 78, 130))
p.add_vital(VitalsReading(date(2026, 3, 3), 135, 86, 74, 125))
p.add_vital(VitalsReading(date(2026, 3, 5), 128, 82, 72, 110))

print("Latest vital:", p.latest_vital())
print("Average systolic:", round(p.average_systolic(), 1))
print("BP control rate:", round(p.bp_control_rate(), 2))
print("Risk flags:", p.risk_flags())


## 5) OOP in action: build a simple follow-up list (registry)

Create multiple patients and generate a “follow-up” list for those with uncontrolled latest BP.


In [ ]:
patients = [
    Patient("P10001", "Alicia", 67, "F", conditions=["Hypertension"], medications=["amlodipine"]),
    Patient("P10002", "Ben", 52, "M", conditions=["Asthma"], medications=[]),
    Patient("P10003", "Carlos", 74, "M", conditions=["Hypertension", "Diabetes"], medications=["metformin", "lisinopril"]),
]

patients[0].add_vital(VitalsReading(date(2026, 3, 5), 146, 92, 80, 118))
patients[1].add_vital(VitalsReading(date(2026, 3, 5), 124, 78, 70, 96))
patients[2].add_vital(VitalsReading(date(2026, 3, 5), 138, 88, 76, 165))

follow_up = []
for pt in patients:
    latest = pt.latest_vital()
    if latest and (not latest.bp_controlled()):
        follow_up.append((pt.patient_id, pt.name, latest.systolic, latest.diastolic, latest.bp_category()))

follow_up


## 6) Mini Exercise (Students)

Add a method to `Patient`:

```python
def has_condition(self, name: str) -> bool:
    ...
```

- Return True if the patient has that condition (case-insensitive)
- Test with:
  - `p.has_condition("hypertension")`
  - `p.has_condition("diabetes")`


In [ ]:
# TODO: Implement Patient.has_condition(...) above, then run:
# print(p.has_condition("hypertension"))
# print(p.has_condition("diabetes"))


## 7) Exercise solution (standalone helper)

In [ ]:
def has_condition_example(patient: Patient, name: str) -> bool:
    return name.strip().lower() in [c.strip().lower() for c in patient.conditions]

print("Has hypertension:", has_condition_example(p, "hypertension"))
print("Has diabetes:", has_condition_example(p, "diabetes"))


## 8) Wrap-up

- **Functions** are great for reusable rules and transformations.
- **Classes** are great for organizing patient-centered logic and building maintainable projects.

Next, you can translate these objects into tables with **pandas** (DataFrames) for cohort analytics.
